In [8]:
    import pyodbc
    import sqlite3
    import pandas as pd


    DB = {
        'servername': r'LAPTOP-VAN-GIJO\SQLEXPRESS',
        'database': 'SourceDataModel'
    }

    export_conn = pyodbc.connect('DRIVER={SQL Server};SERVER=' +
        DB['servername'] +
        ';DATABASE=' +
        DB['database'] +
        ';Trusted_Connection=yes'
    )

    export_cursor = export_conn.cursor()

    def delete_tables():
        tables = ["training", "satisfaction", "satisfaction_type", "course", "returned_item",
                "return_reason", "order_details", "order_header", "order_method", "inventory_levels",
                "product_forecast", "product", "sales_staff", "sales_demographic", "age_group",
                "retailer_contact", "retailer_site", "retailer", "sales_branch", "retailer_headquarters",
                "retailer_segment", "product_type", "product_line", "retailer_type", "country",
                "sales_territory"]  # SSMS-tabellen
        
        for table in tables:
            try:
                export_cursor.execute(f"ALTER TABLE {table} NOCHECK CONSTRAINT ALL")

                export_cursor.execute(f"DELETE FROM {table}")
                export_conn.commit()

                export_cursor.execute(f"ALTER TABLE {table} CHECK CONSTRAINT ALL")
                # print(f"Tabel {table} is leeggemaakt.")
            except Exception as e:
                print(f"Fout bij het leegmaken van de tabel {table}: {e}")
                export_conn.rollback()
        
    delete_tables()

    crm_conn = sqlite3.connect('go_crm_train.sqlite')
    sales_conn = sqlite3.connect('go_sales_train.sqlite')
    staff_conn = sqlite3.connect('go_staff_train.sqlite')

    def move_sales_territory():
        df = pd.read_sql_query("SELECT * FROM sales_territory", crm_conn)
        
        for index, row in df.iterrows():
            try:
                query = f"INSERT INTO sales_territory VALUES ({row['SALES_TERRITORY_CODE']}, '{row['TERRITORY_NAME_EN']}' )"
                export_cursor.execute(query)
            except pyodbc.Error as e:
                print(f"Fout in retailer: {e}")

    def move_country():
        crm_df = pd.read_sql_query("SELECT * FROM country", crm_conn)
        sales_df = pd.read_sql_query("SELECT COUNTRY_CODE, LANGUAGE, CURRENCY_NAME FROM country", sales_conn)
        
        merged_df = pd.merge(crm_df, sales_df, on='COUNTRY_CODE')

        for index, row in merged_df.iterrows():
            try:
                query = f"INSERT INTO country VALUES ({row['COUNTRY_CODE']}, '{row['COUNTRY_EN']}', '{row['LANGUAGE']}','{row['CURRENCY_NAME']}','{row['FLAG_IMAGE']}', {row['SALES_TERRITORY_CODE']} )"
                export_cursor.execute(query)
            except pyodbc.Error as e:
                print(f"Fout in country: {e}")

    def move_retailer_type():
        df = pd.read_sql_query("SELECT * FROM retailer_type", crm_conn)

        for index, row in df.iterrows():
            try:
                query = f"INSERT INTO retailer_type VALUES ({row['RETAILER_TYPE_CODE']}, '{row['RETAILER_TYPE_EN']}' )"
                export_cursor.execute(query)
            except pyodbc.Error as e:
                print(f"Fout in retailer_type: {e}")

    def move_product_line():
        df = pd.read_sql_query("SELECT * FROM product_line", sales_conn)
        
        for index, row in df.iterrows():
            try:
                query = f"INSERT INTO product_line VALUES ({row['PRODUCT_LINE_CODE']}, '{row['PRODUCT_LINE_EN']}' )"
                export_cursor.execute(query)
            except pyodbc.Error as e:
                print(f"Fout in retailer: {e}")

    def move_product_type():
        df = pd.read_sql_query("SELECT * FROM product_type", sales_conn)
        
        for index, row in df.iterrows():
            try:
                query = f"INSERT INTO product_type VALUES ({row['PRODUCT_TYPE_CODE']} , {row['PRODUCT_LINE_CODE']} ,'{row['PRODUCT_TYPE_EN']}' )"
                export_cursor.execute(query)
            except pyodbc.Error as e:
                print(f"Fout in retailer: {e}")

    def move_retailer_segment():
        df = pd.read_sql_query("SELECT * FROM retailer_segment", crm_conn)
        
        for index, row in df.iterrows():
            try:
                query = f"INSERT INTO retailer_segment VALUES ({row['SEGMENT_CODE']}, '{row['LANGUAGE']}', '{row['SEGMENT_NAME']}','{row['SEGMENT_DESCRIPTION']}' )"
                export_cursor.execute(query)
            except pyodbc.Error as e:
                print(f"Fout in retailer_segment: {e}")

    def move_retailer_headquarters():
        df = pd.read_sql_query("SELECT * FROM retailer_headquarters", crm_conn)

        query = """INSERT INTO retailer_headquarters (RETAILER_CODEMR, RETAILER_NAME, ADDRESS1, ADDRESS2, CITY, REGION, COUNTRY_CODE, PHONE, FAX, SEGMENT_CODE)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)"""
        
        for index, row in df.iterrows():
            try:
                export_cursor.execute(query, (row['RETAILER_CODEMR'], row['RETAILER_NAME'], row['ADDRESS1'], row['ADDRESS2'], row['CITY'], row['REGION'], row['COUNTRY_CODE'], row['PHONE'], row['FAX'], row['SEGMENT_CODE']))
            except pyodbc.Error as e:
                print(f"Fout in retailer_headquarters: {e}")

    def move_sales_branch():
        df = pd.read_sql_query("SELECT * FROM sales_branch", staff_conn)
        
        for index, row in df.iterrows():
            try:
                address1 = row['ADDRESS1'].replace("'", "''") if row['ADDRESS1'] is not None else ''
                address2 = row['ADDRESS2'].replace("'", "''") if row['ADDRESS2'] is not None else ''
                city = row['CITY'].replace("'", "''") if row['CITY'] is not None else ''
                region = row['REGION'].replace("'", "''") if row['REGION'] is not None else ''
                query = f"INSERT INTO sales_branch VALUES ({row['SALES_BRANCH_CODE']}, '{address1}', '{address2}','{city}','{region}',{row['COUNTRY_CODE']})"
                export_cursor.execute(query)
            except pyodbc.Error as e:
                print(f"Fout in sales_branch: {e}")

    def move_retailer():
        df = pd.read_sql_query("SELECT * FROM retailer", crm_conn)
        
        for index, row in df.iterrows():
            try:
                retailer_codemr = 'NULL' if pd.isna(row['RETAILER_CODEMR']) else row['RETAILER_CODEMR']
                company_name = row['COMPANY_NAME'].replace("'", "''") if row['COMPANY_NAME'] is not None else ''
                query = f"INSERT INTO retailer VALUES ({row['RETAILER_CODE']}, {retailer_codemr}, '{company_name}', {row['RETAILER_TYPE_CODE']})"
                export_cursor.execute(query)
            except pyodbc.Error as e:
                print(f"Fout in retailer: {e}")

    def move_retailer_site():
        df = pd.read_sql_query("SELECT * FROM retailer_site", crm_conn)
        
        for index, row in df.iterrows():
            try:
                address1 = row['ADDRESS1'].replace("'", "''") if row['ADDRESS1'] is not None else ''
                address2 = row['ADDRESS2'].replace("'", "''") if row['ADDRESS2'] is not None else ''
                city = row['CITY'].replace("'", "''") if row['CITY'] is not None else ''
                region = row['REGION'].replace("'", "''") if row['REGION'] is not None else ''
                query = f"INSERT INTO retailer_site VALUES ({row['RETAILER_SITE_CODE']}, {row['RETAILER_CODE']},'{address1}', '{address2}', '{city}', '{region}', {row['COUNTRY_CODE']}, {row['ACTIVE_INDICATOR']})"
                export_cursor.execute(query)
            except pyodbc.Error as e:
                print(f"Fout in retailer_site: {e}")

    def move_retailer_contact():
        df = pd.read_sql_query("SELECT * FROM retailer_contact", crm_conn)

        for index, row in df.iterrows():
            try:
                retailer_contact_code = 'NULL' if pd.isna(row['RETAILER_CONTACT_CODE']) else row['RETAILER_CONTACT_CODE']
                retailer_site_code = 'NULL' if pd.isna(row['RETAILER_SITE_CODE']) else row['RETAILER_SITE_CODE']
                first_name = row['FIRST_NAME'].replace("'", "''") if row['FIRST_NAME'] is not None else ''
                last_name = row['LAST_NAME'].replace("'", "''") if row['LAST_NAME'] is not None else ''
                job_position_en = row['JOB_POSITION_EN'].replace("'", "''") if row['JOB_POSITION_EN'] is not None else ''
                extension = 'NULL' if pd.isna(row['EXTENSION']) else row['EXTENSION']
                fax = row['FAX'].replace("'", "''") if row['FAX'] is not None else ''
                email = row['E_MAIL'].replace("'", "''") if row['E_MAIL'] is not None else ''
                gender = row['GENDER'].replace("'", "''") if row['GENDER'] is not None else ''
                
                query = f"INSERT INTO retailer_contact VALUES ({retailer_contact_code}, {retailer_site_code}, '{first_name}', '{last_name}', '{job_position_en}', {extension}, '{fax}', '{email}', '{gender}')"
                export_cursor.execute(query)
            except pyodbc.Error as e:
                print(f"Fout in retailer_contact: {e}")

    def move_age_group():
        df = pd.read_sql_query("SELECT * FROM age_group", crm_conn)
        
        for index, row in df.iterrows():
            try:
                query = f"INSERT INTO age_group VALUES ({row['AGE_GROUP_CODE']}, {row['UPPER_AGE']}, {row['LOWER_AGE']})"
                export_cursor.execute(query)
            except pyodbc.Error as e:
                print(f"Fout in age_group: {e}")

    def move_sales_demographic():
        df = pd.read_sql_query("SELECT * FROM sales_demographic", crm_conn)
        
        for index, row in df.iterrows():
            try:
                query = f"INSERT INTO sales_demographic VALUES ({row['DEMOGRAPHIC_CODE']}, {row['RETAILER_CODEMR']}, {row['AGE_GROUP_CODE']}, {row['SALES_PERCENT']} )"
                export_cursor.execute(query)
            except pyodbc.Error as e:
                print(f"Fout in sales_demographic: {e}")

    def move_sales_staff():
        df = pd.read_sql_query("SELECT * FROM sales_staff", staff_conn)

        for index, row in df.iterrows():
            try:
                sales_staff_code = 'NULL' if pd.isna(row['SALES_STAFF_CODE']) else row['SALES_STAFF_CODE']
                first_name = row['FIRST_NAME'].replace("'", "''") if row['FIRST_NAME'] is not None else ''
                last_name = row['LAST_NAME'].replace("'", "''") if row['LAST_NAME'] is not None else ''
                position_en = row['POSITION_EN'].replace("'", "''") if row['POSITION_EN'] is not None else ''
                work_phone = row['WORK_PHONE'].replace("'", "''") if row['WORK_PHONE'] is not None else ''
                extension = 'NULL' if pd.isna(row['EXTENSION']) else row['EXTENSION']
                fax = row['FAX'].replace("'", "''") if row['FAX'] is not None else ''
                email = row['EMAIL'].replace("'", "''") if row['EMAIL'] is not None else ''
                date_hired = row['DATE_HIRED'].replace("'", "''") if row['DATE_HIRED'] is not None else ''
                sales_branch_code = 'NULL' if pd.isna(row['SALES_BRANCH_CODE']) else row['SALES_BRANCH_CODE']
                
                query = f"INSERT INTO sales_staff (SALES_STAFF_CODE, FIRST_NAME, LAST_NAME, POSITION_EN, WORK_PHONE, EXTENSION, FAX, EMAIL, DATE_HIRED, SALES_BRANCH_CODE) VALUES ({sales_staff_code}, '{first_name}', '{last_name}', '{position_en}', '{work_phone}', {extension}, '{fax}', '{email}', '{date_hired}', {sales_branch_code})"
                export_cursor.execute(query)
            except pyodbc.Error as e:
                print(f"Fout in sales_staff (first pass): {e}")
        
        for index, row in df.iterrows():
            try:
                if not pd.isna(row['MANAGER_CODE']):
                    manager_code = 'NULL' if pd.isna(row['MANAGER_CODE']) else row['MANAGER_CODE']
                    sales_staff_code = 'NULL' if pd.isna(row['SALES_STAFF_CODE']) else row['SALES_STAFF_CODE']
                    query = f"UPDATE sales_staff SET MANAGER_CODE = {manager_code} WHERE SALES_STAFF_CODE = {sales_staff_code}"
                    export_cursor.execute(query)
            except pyodbc.Error as e:
                print(f"Fout in sales_staff (second pass): {e}")

    def move_product():
        df = pd.read_sql_query("SELECT * FROM product", sales_conn)
        
        for index, row in df.iterrows():
            try:
                product_number = row['PRODUCT_NUMBER']
                introduction_date = row['INTRODUCTION_DATE']
                product_type_code = row['PRODUCT_TYPE_CODE']
                production_cost = row['PRODUCTION_COST']
                margin = row['MARGIN']
                product_image = row['PRODUCT_IMAGE'].replace("'", "''") if row['PRODUCT_IMAGE'] is not None else ''
                language = row['LANGUAGE'].replace("'", "''") if row['LANGUAGE'] is not None else ''
                product_name = row['PRODUCT_NAME'].replace("'", "''") if row['PRODUCT_NAME'] is not None else ''
                description = row['DESCRIPTION'].replace("'", "''") if row['DESCRIPTION'] is not None else ''
                
                query = f"INSERT INTO product (PRODUCT_NUMBER, INTRODUCTION_DATE, PRODUCT_TYPE_CODE, PRODUCTION_COST, MARGIN, PRODUCT_IMAGE, LANGUAGE, PRODUCT_NAME, DESCRIPTION) VALUES ({product_number}, '{introduction_date}', {product_type_code}, {production_cost}, {margin}, '{product_image}', '{language}', '{product_name}', '{description}')"
                export_cursor.execute(query)
            except pyodbc.Error as e:
                print(f"Fout bij product {product_number}: {e}")

    def move_product_forecast():
        df = pd.read_csv("product_forecast_train.csv")
        forecast_code = 1  

        for index, row in df.iterrows():
            try:
                query = f"INSERT INTO product_forecast VALUES ({forecast_code}, {row['PRODUCT_NUMBER']}, {row['EXPECTED_VOLUME']}, {row['YEAR']}, {row['MONTH']})"
                export_cursor.execute(query)
                forecast_code += 1  
            except pyodbc.Error as e:
                print(f"Fout in product_forecast: {e}")

    def move_inventory_levels():
        df = pd.read_csv("inventory_levels_train.csv")
        for index, row in df.iterrows():
            try:
                query = f"INSERT INTO Inventory_levels VALUES ({row['INVENTORY_CODE']}, {row['PRODUCT_NUMBER']}, {row['INVENTORY_COUNT']}, {row['INVENTORY_YEAR']}, '{row['INVENTORY_MONTH']}')"
                export_cursor.execute(query)
            except pyodbc.Error as e:
                print(f"Fout in inventory_levels: {e}")

    def move_order_method():
        df = pd.read_sql_query("SELECT * FROM order_method", sales_conn)
        
        for index, row in df.iterrows():
            try:
                query = f"INSERT INTO order_method VALUES ({row['ORDER_METHOD_CODE']}, '{row['ORDER_METHOD_EN']}' )"
                export_cursor.execute(query)
            except pyodbc.Error as e:
                print(f"Fout in retailer: {e}")

    def move_order_header():
        df = pd.read_sql_query("SELECT * FROM order_header", sales_conn)
        
        for index, row in df.iterrows():
            try:
                retailer_name = row['RETAILER_NAME'].replace("'", "''") if row['RETAILER_NAME'] is not None else ''
                query = f"INSERT INTO order_header VALUES ({row['ORDER_NUMBER']}, '{retailer_name}', {row['RETAILER_SITE_CODE']}, {row['RETAILER_CONTACT_CODE']}, {row['SALES_STAFF_CODE']}, {row['SALES_BRANCH_CODE']}, '{row['ORDER_DATE']}', {row['ORDER_METHOD_CODE']})"
                export_cursor.execute(query)
            except pyodbc.Error as e:
                print(f"Fout in order_header: {e}")

    def move_order_details():
        df = pd.read_sql_query("SELECT * FROM order_details", sales_conn)
        
        for index, row in df.iterrows():
            try:
                query = f"INSERT INTO order_details VALUES ({row['ORDER_DETAIL_CODE']},{row['ORDER_NUMBER']}, {row['PRODUCT_NUMBER']}, {row['QUANTITY']}, {row['UNIT_COST']}, {row['UNIT_PRICE']}, {row['UNIT_SALE_PRICE']})"
                export_cursor.execute(query)
            except pyodbc.Error as e:
                print(f"Fout in order_details: {e}")

    def move_return_reason():
        df = pd.read_sql_query("SELECT * FROM return_reason", sales_conn)
        
        for index, row in df.iterrows():
            try:
                query = f"INSERT INTO return_reason VALUES ({row['RETURN_REASON_CODE']}, '{row['RETURN_DESCRIPTION_EN']}' )"
                export_cursor.execute(query)
            except pyodbc.Error as e:
                print(f"Fout in return_reason: {e}")

    def move_returned_item():
        df = pd.read_sql_query("SELECT * FROM returned_item", sales_conn)
        
        for index, row in df.iterrows():
            try:
                query = f"INSERT INTO returned_item VALUES ({row['RETURN_CODE']}, '{row['RETURN_DATE']}', {row['ORDER_DETAIL_CODE']}, {row['RETURN_REASON_CODE']}, {row['RETURN_QUANTITY']})"
                export_cursor.execute(query)
            except pyodbc.Error as e:
                print(f"Fout in returned_item: {e}")



    def move_course():
        df = pd.read_sql_query("SELECT * FROM course", staff_conn)
        
        for index, row in df.iterrows():
            try:
                query = f"INSERT INTO course VALUES ({row['COURSE_CODE']}, '{row['COURSE_DESCRIPTION']}' )"
                export_cursor.execute(query)
            except pyodbc.Error as e:
                print(f"Fout in retailer: {e}")

    def move_satisfaction_type():
        df = pd.read_sql_query("SELECT * FROM satisfaction_type", staff_conn)
        
        for index, row in df.iterrows():
            try:
                query = f"INSERT INTO satisfaction_type VALUES ({row['SATISFACTION_TYPE_CODE']}, '{row['SATISFACTION_TYPE_DESCRIPTION']}' )"
                export_cursor.execute(query)
            except pyodbc.Error as e:
                print(f"Fout in retailer: {e}")


    def move_satisfaction():
        df = pd.read_sql_query("SELECT * FROM satisfaction", staff_conn)
        
        for index, row in df.iterrows():
            try:
                query = f"INSERT INTO satisfaction VALUES ({row['YEAR']}, {row['SALES_STAFF_CODE']}, {row['SATISFACTION_TYPE_CODE']})"
                export_cursor.execute(query)
            except pyodbc.Error as e:
                print(f"Fout in satisfaction: {e}")


    def move_training():
        df = pd.read_sql_query("SELECT * FROM training", staff_conn)

        for index, row in df.iterrows():
            try:
                query = f"INSERT INTO training VALUES ({row['YEAR']}, '{row['SALES_STAFF_CODE']}', {row['COURSE_CODE']} )"
                export_cursor.execute(query)
            except pyodbc.Error as e:
                print(f"Fout in training: {e}")




    move_sales_territory()
    move_country()
    move_retailer_type()
    move_product_line()
    move_product_type()
    move_retailer_segment()
    move_retailer_headquarters()
    move_sales_branch()
    move_retailer()
    move_retailer_site()
    move_retailer_contact()
    move_age_group()
    move_sales_demographic()
    move_sales_staff()
    move_product()
    move_product_forecast()
    move_inventory_levels()
    move_order_method()
    move_order_header()
    move_order_details()
    move_return_reason()
    move_returned_item()
    move_course()
    move_satisfaction_type()
    move_satisfaction()
    move_training()

    crm_conn.close()
    sales_conn.close()
    staff_conn.close()

    export_conn.commit()
    export_cursor.close()
    export_conn.close()